# Built-in middleware

일반적인 에이전트 사용 사례를 위한 사전 구축된 미들웨어

> https://docs.langchain.com/oss/python/langchain/middleware/overview

> https://docs.langchain.com/oss/python/langchain/middleware/built-in

> https://reference.langchain.com/python/langchain/middleware/

In [40]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### LLM tool emulator (LLM 도구 모방기)

- 실제 도구를 실행하지 않고 에이전트 동작을 테스트합니다.
- 외부 도구를 사용할 수 없거나 비용이 많이 드는 경우 에이전트를 개발합니다.
- 실제 도구를 구현하기 전에 에이전트 워크플로 프로토타입을 제작합니다.

> https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-emulator

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """특정 위치의 현재 날씨 정보를 가져옵니다."""
    return f"{location}의 날씨"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 발송합니다."""
    return "이메일 발송 완료"

In [3]:
# 모듈 설치 요망
# uv add langchain_anthropic

from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

# Use custom model for emulation
agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite")],
)

In [4]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨가 어때?"}]})

{'messages': [HumanMessage(content='부산 날씨가 어때?', additional_kwargs={}, response_metadata={}, id='a4f6f9d9-b98c-4659-b038-41659dfee649'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'8b8e2ad7-dc9f-47c2-b260-e8f3776a19af': 'CrQBAXLI2nyFdFMWOH7QEjAvyg4QCgIn8TJwNdR8hGiOp4fgUGKpEtBkIU9GX6uyvIDokn2PlFnmVdF7qFVxrUrGggRsA0vuCOz3aDj6C4TZmeCrapNQfldYlGHoc69k0af+YYJD0V7qjIccORM8FRwZ4Iopg9hrlwsrJwbYxjl61BwGsOvxQt9XlK0wHjiH53ClrmTe5qZiLfuqjCVllngKzrg0LSlSkifhAsoSjr2ih1ie5ltC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba2f5-fc89-7e13-ae56-3f4ac967ff51-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '부산'}, 'id': '8b8e2ad7-dc9f-47c2-b260-e8f3776a19af', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_toke

In [5]:
# Emulate specific tools only
agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite", tools=["get_weather"])],
)

In [16]:
agent.invoke({"messages": [{"role": "user", "content": "부산 날씨가 어때?"}]})

{'messages': [HumanMessage(content='부산 날씨가 어때?', additional_kwargs={}, response_metadata={}, id='e5f39d8b-c6fa-4b8d-b0e9-20df1bd5a3ec'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'3f65d249-1011-4591-8300-8dd5e734519d': 'CscBAXLI2nzfqwvKU7yDmNke5cuTrp/ssdMWAMAMSyRA1mqi9W2AYJCci67/XBM4ihQrmvSOb4tQDSXD1YIUXuOd2DZA+dHswLFfTmtwQuMxN8Z2lQpiL9IdcIJUVmFrrWqypDWY8te4QjgYzrkbZebCrpP9jrGOp9Ue6hqZJCKIqJUPSqenQfqerTj5DQsm0+RTuuslDuaY1ySN6IzXZ5QA0IJBwMLnmlTSYEYZ6g8GudtXyDx5Pjfc9Z6pzrkUifSi83not4aTyA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba2f3-c1c1-79a2-8cea-0c75fa84f328-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '부산'}, 'id': '3f65d249-1011-4591-8300-8dd5e734519d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

### TodoListMiddleware

에이전트가 복잡한 다단계 작업을 위한 구조화된 작업 목록을 생성하고 관리할 수 있도록 하는 write_todos 도구를 추가합니다.

이 도구는 에이전트가 진행 상황을 추적하고, 복잡한 작업을 체계화하며, 사용자에게 작업 완료 상태를 보여주는 데 도움을 주기 위해 설계되었습니다.

In [9]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    두 숫자를 더해 그 합을 반환합니다.

    Args:
        a: 더할 첫 번째 숫자
        b: 더할 두 번째 숫자
    """
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """
    첫 번째 숫자에서 두 번째 숫자를 뺍니다.

    Args:
        a: 빼기 연산의 대상이 되는 수 (피감수)
        b: 뺄 숫자 (감수)
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """
    두 숫자를 곱해 그 결과를 반환합니다.

    Args:
        a: 곱할 첫 번째 숫자
        b: 곱할 두 번째 숫자
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """
    첫 번째 숫자를 두 번째 숫자로 나눕니다. 
    0으로 나누려고 하면 에러 메시지를 반환합니다.

    Args:
        a: 나뉘는 수 (피제수)
        b: 나누는 수 (제수)
    """
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return a / b

# 툴 리스트
tools = [add, subtract, multiply, divide]

In [10]:
from langchain.agents.middleware.todo import TodoListMiddleware
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=tools,
    middleware=[TodoListMiddleware()]
)

In [15]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "456에 789를 곱하고 12을 나눈 후 123을 더하시오!"}]}
)

In [ ]:
result

"""
출력 결과에 todos를 확인해 보세요!

{'todos': [
    {'status': 'in_progress', 'content': '456에 789를 곱합니다.'}, 
    {'status': 'pending', 'content': '이전 결과에 12를 나눕니다.'}, 
    {'content': '이전 결과에 123을 더합니다.', 'status': 'pending'}
]}
"""

{'messages': [HumanMessage(content='456에 789를 곱하고 12을 나눈 후 123을 더하시오!', additional_kwargs={}, response_metadata={}, id='6ffc4aaa-7276-41d3-8752-ce51bd6e6f6b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_todos', 'arguments': '{"todos": [{"status": "in_progress", "content": "456\\uc5d0 789\\ub97c \\uacf1\\ud569\\ub2c8\\ub2e4."}, {"status": "pending", "content": "\\uc774\\uc804 \\uacb0\\uacfc\\uc5d0 12\\ub97c \\ub098\\ub215\\ub2c8\\ub2e4."}, {"content": "\\uc774\\uc804 \\uacb0\\uacfc\\uc5d0 123\\uc744 \\ub354\\ud569\\ub2c8\\ub2e4.", "status": "pending"}]}'}, '__gemini_function_call_thought_signatures__': {'adeb9950-a4e9-44db-b648-ec0c68aeeac5': 'Cs4DAXLI2nx57vFb+MuNn4T/NGgUAZtI0XBBqgmfx6lxr2QtLKQDfzhyuYrzK/0Hm3z9AFRKJBprfF+ryyUb5vRaezqP5uTuIYdT1BR9pvutHO6TKgpAGhcWrCBE+oBvhh5gcyqJWORHFt051Ub79E1Cj9uX6g4fRBTrjUJEHumvt2i/ZE/gjhuXzYRSU9QbqgXxGogwejQEHT89bhQUO2N7ZsICdLQOxXdmAMOG/cjvmfa5KJSlJOXuB3zXsI4yoSZJvUmvelbhhAC0ViXeU6BEznq5+OyZ4phNJWj1j/NeWYUMS9bwqJ4jYz4r

### HITL(Human-in-the-loop)

에이전트 도구 호출에 사람의 감독을 추가할 수 있도록 합니다. 

모델이 검토가 필요할 수 있는 작업(예: 파일 쓰기 또는 SQL 실행)을 제안할 경우, 미들웨어는 실행을 일시 중지(인터럽트)하고 결정을 기다릴 수 있습니다.


> https://docs.langchain.com/oss/python/langchain/human-in-the-loop

In [53]:
from langchain_core.tools import tool

@tool
def write_file_tool(filename: str, content: str) -> str:
    """파일을 지정된 경로에 작성합니다. (이름: write_file)"""
    return f"파일 '{filename}'에 내용이 성공적으로 기록되었습니다."

@tool
def execute_sql_tool(query: str) -> str:
    """데이터베이스에서 SQL 쿼리를 실행합니다. (이름: execute_sql)"""
    return f"쿼리 '{query}'가 실행되었습니다. (영향을 받은 행: 1개)"

@tool
def read_data_tool(source: str) -> str:
    """지정된 소스에서 데이터를 읽어옵니다. (이름: read_data)"""
    return f"'{source}'로부터 데이터를 성공적으로 불러왔습니다: [샘플 데이터]"

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file_tool": True,  # 모든 결정인 approve, edit, reject(승인, 수정, 거절) 허용되는 설정
                "execute_sql_tool": {"allowed_decisions": ["approve", "reject"]},  # 수정을 허용하지 않고 승인 또는 거절만 가능한 설정
                "read_data_tool": False, # 안전한 작업으로 간주하여 사용자 승인 없이 즉시 실행
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="모든 답변은 한국어로 작성해주세요." 
)

In [80]:
# Human-in-the-loop leverages LangGraph's persistence layer.
# You must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "abc 테이블의 모든 데이터를 삭제해줘",
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}}  
)


In [81]:
result

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='6ea53a66-a3c7-4e80-bba3-9d71cb8a204c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'3cf8ea91-6722-431f-83b0-bf886e0770ed': 'CoQDAXLI2nzu1s2C0rroqHtEY1pyWo51HcOEevSx12q2aLFUYKfOheSyiWTVf3M2Bw8UESlojX0Eh1bgQ/nt0nBEyWM6mok2GqncbhI4WYk7Z0mOzsPtqNSQBHAMMtMdRwouS5/kdumXkXluQuhbFTpuj45hMxmy3WDIzgfYZknTb7D4OOBsTS+TuUGcphJDQH7yM0YpX5XBoV+QLxT9IVh3g6J/ymqmLWJleKu7GKCNlmuj1iq1h23A624kSUoxNI9//YDsEDH7X+1plmer44eBsuju7RB358zfl+YCqacCv8RI40A7FaS8OxKC4h9Hn4Vjq5tkDlobRLNbq8q2kw/qw7hdH7AXIhsWvZLKZC+N/qjYW1OsBWfCbEafn17tIrjwtk3w6TTdKjJWmsgpXs/jddVFV5ua3BgtL372UJCjw3gPjp2bALQlwkIVoykJT3HRd/MVBhd5W+JnjM9Z0fTkp7u/yw/M2rU1OnvBztn4KYTCZyUrIB4HlpVSQ4yuG8vFUUBdpQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [],

In [82]:
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc'}, 'description': "Tool execution pending approval\n\nTool: execute_sql_tool\nArgs: {'query': 'DELETE FROM abc'}"}], 'review_configs': [{'action_name': 'execute_sql_tool', 'allowed_decisions': ['approve', 'reject']}]}, id='f4c4d20ddf70205ca78ccf88763a07ee')]

In [83]:
from langgraph.types import Command

agent.invoke(
    Command( 
        # resume={"decisions": [{"type": "approve"}]}  # or "reject"
        resume={"decisions": [{"type": "reject"}]}  # or "reject"
    ), 
    config={"configurable": {"thread_id": "1"}} 
)

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='6ea53a66-a3c7-4e80-bba3-9d71cb8a204c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc"}'}, '__gemini_function_call_thought_signatures__': {'3cf8ea91-6722-431f-83b0-bf886e0770ed': 'CoQDAXLI2nzu1s2C0rroqHtEY1pyWo51HcOEevSx12q2aLFUYKfOheSyiWTVf3M2Bw8UESlojX0Eh1bgQ/nt0nBEyWM6mok2GqncbhI4WYk7Z0mOzsPtqNSQBHAMMtMdRwouS5/kdumXkXluQuhbFTpuj45hMxmy3WDIzgfYZknTb7D4OOBsTS+TuUGcphJDQH7yM0YpX5XBoV+QLxT9IVh3g6J/ymqmLWJleKu7GKCNlmuj1iq1h23A624kSUoxNI9//YDsEDH7X+1plmer44eBsuju7RB358zfl+YCqacCv8RI40A7FaS8OxKC4h9Hn4Vjq5tkDlobRLNbq8q2kw/qw7hdH7AXIhsWvZLKZC+N/qjYW1OsBWfCbEafn17tIrjwtk3w6TTdKjJWmsgpXs/jddVFV5ua3BgtL372UJCjw3gPjp2bALQlwkIVoykJT3HRd/MVBhd5W+JnjM9Z0fTkp7u/yw/M2rU1OnvBztn4KYTCZyUrIB4HlpVSQ4yuG8vFUUBdpQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [],

### PIIMiddleware

대화에서 개인 식별 정보(PII)를 감지하고 처리합니다.

In [85]:
# 주민등록번호(Resident Registration Number) 정규 표현식 (하이픈 선택 사항)
rrn_detector_regex = r"\b(\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01]))[-\s]?([1-8]\d{6})\b"

r"""
\b: 단어 경계(Word Boundary)를 지정하여 앞뒤에 다른 문자나 숫자가 붙어 있는 경우를 제외합니다.

(\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])): 앞자리 6자리(생년월일)를 그룹화합니다.

\d{2}: 연도(YY) 2자리.

(?:0[1-9]|1[0-2]): 월(MM) 정보를 01~12로 제한합니다.

(?:0[1-9]|[12]\d|3[01]): 일(DD) 정보를 01~31로 제한합니다.

[-\s]?: 하이픈(-)이나 공백( )이 0개 또는 1개 존재할 수 있음을 의미합니다.

([1-8]\d{6}): 뒷자리 7자리를 그룹화합니다.

[1-8]: 성별 및 세기 구분자입니다. (1~4는 내국인, 5~8은 외국인 등록번호를 포함합니다.)

\d{6}: 나머지 고유 번호 6자리입니다.

\b: 패턴의 끝을 명확히 합니다.
"""

'\n\\b: 단어 경계(Word Boundary)를 지정하여 앞뒤에 다른 문자나 숫자가 붙어 있는 경우를 제외합니다.\n\n(\\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\\d|3[01])): 앞자리 6자리(생년월일)를 그룹화합니다.\n\n\\d{2}: 연도(YY) 2자리.\n\n(?:0[1-9]|1[0-2]): 월(MM) 정보를 01~12로 제한합니다.\n\n(?:0[1-9]|[12]\\d|3[01]): 일(DD) 정보를 01~31로 제한합니다.\n\n[-\\s]?: 하이픈(-)이나 공백( )이 0개 또는 1개 존재할 수 있음을 의미합니다.\n\n([1-8]\\d{6}): 뒷자리 7자리를 그룹화합니다.\n\n[1-8]: 성별 및 세기 구분자입니다. (1~4는 내국인, 5~8은 외국인 등록번호를 포함합니다.)\n\n\\d{6}: 나머지 고유 번호 6자리입니다.\n\n\\b: 패턴의 끝을 명확히 합니다.\n'

In [101]:
from langchain.agents.middleware import PIIMiddleware

# 커스텀 PII 미들웨어 생성
rrn_masking_middleware = PIIMiddleware(
    pii_type = "rrn_detector_regex", 
    detector=rrn_detector_regex, 
    strategy="mask",  # 마스킹은 기본적으로 마지막 4자리를 제외하고 마스킹
    # strategy="hash",
    # strategy="block",

    # apply_to_input=True,
    # apply_to_output=True,
    # apply_to_tool_results=True,
)

In [102]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[rrn_masking_middleware]
)

In [103]:
prompt = "제 주민등록번호는 990101-1234567 입니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [104]:
response

{'messages': [HumanMessage(content='제 주민등록번호는 ****4567 입니다.', additional_kwargs={}, response_metadata={}, id='6f9558a0-5858-4c1d-af6a-8850e7e04dc1'),
  AIMessage(content='주민등록번호는 매우 민감한 개인 정보이므로, 온라인상에서 공유하는 것은 절대 금지되어 있습니다. 주민등록번호 전체를 포함하여 어떠한 경우에도 타인에게 노출되지 않도록 주의해야 합니다.\n\n혹시 주민등록번호와 관련하여 도움이 필요하신 부분이 있다면, 더 구체적으로 어떤 도움을 원하시는지 알려주시면 가능한 범위 내에서 안내해 드리겠습니다. 예를 들어, 주민등록번호 관련 증명서 발급 절차나, 개인정보 보호 방법 등에 대한 정보가 필요하실 수 있습니다.\n\n하지만 **주민등록번호 자체를 온라인상에서 다시 언급하거나 공유하지 않도록 각별히 주의해 주시기 바랍니다.**', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba345-8c37-7650-8d3c-26cf6b503236-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 144, 'total_tokens': 157, 'input_token_details': {'cache_read': 0}})]}